In [1]:
!pip install -q -U transformers peft bitsandbytes accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 82.0 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 31.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 18.6 MB/s eta 0:00:00


In [2]:
import torch
import gc
import pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
from tqdm import tqdm
import os

PROMPTS = [
    # --- Tech & Lavoro ---
    "Make a joke about work calls.",
    "Tell me a joke about software engineers.",
    "Make a dark humor joke about Artificial Intelligence taking our jobs.",
    "Tell me a joke about trying to fix a printer.",
    "Make a sarcastic comment about LinkedIn influencers.",
    "Tell me a joke about zoom meetings.",
    "Make a joke about working from home.",
    "Tell me a joke about forgetting your passwords.",
    "Make a sarcastic comment about autocorrect ruining texts.",
    
    # --- Satira, Economia & Società ---
    "Tell me a joke about politicians.",
    "Make a sarcastic comment about modern dating apps.",
    "Tell me a joke about vegans.",
    "Make a joke about climate change.",
    "Tell me a joke about inflation and prices going up.",
    "Make a sarcastic comment about cryptocurrency bros.",
    "Tell me a joke about the healthcare system.",
    "Make a joke about modern art.",
    "Tell me a sarcastic joke about social media algorithms.",
    
    # --- Stereotipi & Nazioni (Test per Prompt Bleed) ---
    "Tell me a joke about Italians.",
    "Make a joke about British food.",
    "Tell me a joke about Americans.",
    "Make a joke about French people.",
    "Tell me a joke about Germans and their efficiency.",
    "Make a sarcastic comment about tourists.",
    "Tell me a joke about Canadians apologizing.",
    "Make a joke about Australian wildlife.",
    
    # --- Attualità, VIP & Pop Culture ---
    "Tell me a joke about Elon Musk.",
    "Make a joke about Mark Zuckerberg's Metaverse.",
    "Make a sarcastic comment about billionaires going to space.",
    "Tell me a joke about Apple releasing a new iPhone that looks exactly the same.",
    "Make a joke about having too many streaming service subscriptions.",
    "Tell me a joke about political TV debates.",
    "Make a sarcastic comment about reality TV shows.",
    
    # --- Vita Quotidiana (Observational Comedy) ---
    "Make a joke about going to the gym in January.",
    "Tell me a joke about being broke.",
    "Make a sarcastic comment about family dinners.",
    "Tell me a joke about sleeping 8 hours.",
    "Tell me a joke about trying to assemble IKEA furniture.",
    "Make a sarcastic comment about public transport.",
    "Tell me a joke about forgetting someone's name immediately after meeting them.",
    "Make a joke about going grocery shopping while hungry.",
    "Tell me a sarcastic joke about starting a new diet.",
    "Make a sarcastic comment about people who clap when the airplane lands.",
    
    # --- Black Humor & Absurd ---
    "Make a dark humor joke about going to the doctor.",
    "Tell me a joke about my bank account.",
    "Make a joke about the housing market.",
    "Make a dark humor joke about student loans.",
    "Tell me a dark joke about getting older.",
    "Make a sarcastic comment about the end of the world.",
    "Tell me a dark humor joke about marriage."
]

bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16)
BASE_PATH = "/kaggle/input/datasets/silvioliparoti/aligned-models-fin"
CSV_PATH = "dataset_valutazione.csv"

# Inizializza il salvataggio
results = {"Prompt": PROMPTS}
pd.DataFrame(results).to_csv(CSV_PATH, index=False)

def salva_colonna(nome_colonna, lista_battute):
    df = pd.read_csv(CSV_PATH)
    df[nome_colonna] = lista_battute
    df.to_csv(CSV_PATH, index=False)
    print(f" Checkpoint salvato per {nome_colonna}!")

def generate_for_adapter(base_model_id, tokenizer, adapter_path, prompts, sys_prompt, is_mistral=False):
    print(f"\n---> Caricamento Modello Base: {base_model_id}")
    base_model = AutoModelForCausalLM.from_pretrained(base_model_id, quantization_config=bnb_config, device_map="auto")
    
    print(f"---> Applicazione Adattatore: {adapter_path}")
    model = PeftModel.from_pretrained(base_model, adapter_path)
    
    generations = []
    
    for user_prompt in tqdm(prompts, desc="Generazione"):
        # GESTIONE DIFFERENZIATA: Text-Completion per Mistral, Chat per Zephyr
        if is_mistral:
            # USIAMO IL FORMATO ESATTO DELLA DEMO PPO_V1
            input_text = f"<s>[SYSTEM] \n {sys_prompt} \n[USER] {user_prompt} . \n [ASSISTANT]</s>"
        else:
            messages = [{"role": "system", "content": sys_prompt}, {"role": "user", "content": user_prompt}]
            input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
            
        inputs = tokenizer(input_text, return_tensors="pt").to("cuda")
        input_length = inputs["input_ids"].shape[1] 
        
        with torch.no_grad():
            outputs = model.generate(
                **inputs, 
                max_new_tokens=60 if is_mistral else 100, 
                do_sample=True, temperature=0.7, top_p=0.90, repetition_penalty=1.15, pad_token_id=tokenizer.eos_token_id
            )
            
        new_tokens = outputs[0][input_length:]
        response = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
        
        # LA TAGLIOLA DIFFERENZIATA
        if is_mistral:
            if "Topic:" in response: response = response.split("Topic:")[0].strip()
            if "Joke:" in response: response = response.split("Joke:")[0].strip()
            if "\n\n" in response: response = response.split("\n\n")[0].strip()
        else:
            stop_phrases = ["Or how about", "Here's another", "Alternatively", "(Ba-dum-tish)", "(Boom tsh)"]
            for phrase in stop_phrases:
                if phrase in response:
                    response = response.split(phrase)[0].strip()
                    
            preambles_inline = ["Sure thing!", "Here it is:", "Sure,", "Here is one:", "Certainly!", "Sure, here's another one:"]
            for p in preambles_inline:
                if response.lower().startswith(p.lower()):
                    response = response[len(p):].strip()

        if not response:
            response = "No joke generated."
            
        generations.append(response)
        
    # DISTRUZIONE TOTALE: Nessun accavallamento tra i modelli!
    del model
    del base_model
    torch.cuda.empty_cache()
    gc.collect()
    return generations


In [3]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
import torch
from tqdm import tqdm

def generate_for_base_model(base_model_id, tokenizer, prompts, system_prompt):
    print(f"  -> Caricamento Modello Base (Senza Adapter): {base_model_id}")
    
    # Configurazione per caricarlo leggero in 4-bit (come fai per gli altri)
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
    )
    
    model = AutoModelForCausalLM.from_pretrained(
        base_model_id, 
        quantization_config=bnb_config, 
        device_map="auto"
    )
    model.eval()
    
    battute_generate = []
    
    with torch.no_grad():
        for user_prompt in tqdm(prompts, desc="Generazione Zero-Shot"):
            # Applica il template di Zephyr
            messages = [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt},
            ]
            prompt_formattato = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
            inputs = tokenizer(prompt_formattato, return_tensors="pt").to("cuda")
            
            outputs = model.generate(
                **inputs,
                max_new_tokens=100,
                temperature=0.7,
                do_sample=True,
                pad_token_id=tokenizer.eos_token_id
            )
            
            # Estrai solo la risposta generata
            input_length = inputs.input_ids.shape[1]
            risposta = tokenizer.decode(outputs[0][input_length:], skip_special_tokens=True).strip()
            battute_generate.append(risposta)
            
    del model
    torch.cuda.empty_cache()
    gc.collect()
    
    return battute_generate

In [4]:
# --- IL MASTER PROMPT DEFINITIVO (UGUALE PER TUTTI I MODELLI) ---
# È STATO CREATO DAL RAGGRUPPAMENTO DEI VARI PROMPT USATI NELLE DEMO DELLE VARIE VERSIONI

MASTER_SYSTEM_PROMPT = (
    "You are a versatile comedy writer and a witty, cynical stand-up comedian. "
    "Your goal is to make the user laugh with exactly ONE short, clever joke about the provided topic. "
    "Depending on the topic, you can deliver a sharp one-liner, a cynical observation, or a dark humor statement. "
    "Keep it brief (one or two lines at most), be direct, and end with a clever twist. "
    "Do not explain the joke and stop immediately after the punchline. Make it funny!"
)

In [8]:

print(" FASE 1: Mistral-7B...")
PATH_PPO_V1 = f"{BASE_PATH}/mistral_aligned_PPO_v1"
tokenizer_mistral = AutoTokenizer.from_pretrained(PATH_PPO_V1)
tokenizer_mistral.pad_token = tokenizer_mistral.eos_token

# is_mistral=True
battute = generate_for_adapter("mistralai/Mistral-7B-Instruct-v0.2", tokenizer_mistral, PATH_PPO_V1, PROMPTS, MASTER_SYSTEM_PROMPT, is_mistral=True)
salva_colonna("PPO_v1", battute)

del tokenizer_mistral
gc.collect()


print("\n FASE 2: Zephyr-7B...")
PATH_PPO_V2 = f"{BASE_PATH}/zephyr_aligned_PPO_v2"
tokenizer_zephyr = AutoTokenizer.from_pretrained(PATH_PPO_V2)
tokenizer_zephyr.pad_token = tokenizer_zephyr.eos_token
BASE_ZEPHYR_ID = "HuggingFaceH4/zephyr-7b-beta"

# Zephyr usa il MASTER_SYSTEM_PROMPT
# 1. Zephyr PPO_v2
battute = generate_for_adapter(BASE_ZEPHYR_ID, tokenizer_zephyr, PATH_PPO_V2, PROMPTS, MASTER_SYSTEM_PROMPT)
salva_colonna("PPO_v2", battute)

# 2. Zephyr DPO_v1
PATH_DPO_V1 = f"{BASE_PATH}/zephyr_aligned_DPO_v1/Zephyr_Manual_DPO_Adapter"
battute = generate_for_adapter(BASE_ZEPHYR_ID, tokenizer_zephyr, PATH_DPO_V1, PROMPTS, MASTER_SYSTEM_PROMPT)
salva_colonna("DPO_v1", battute)

# 3. Zephyr DPO_v2.1
PATH_DPO_V2_1 = f"{BASE_PATH}/zephyr_aligned_DPO_v2_1/Zephyr_Manual_DPO_Adapter_v2_1"
battute = generate_for_adapter(BASE_ZEPHYR_ID, tokenizer_zephyr, PATH_DPO_V2_1, PROMPTS, MASTER_SYSTEM_PROMPT)
salva_colonna("DPO_v2.1", battute)

# 4. Zephyr DPO_v2.2
PATH_DPO_V2_2 = "/kaggle/input/datasets/silvioliparoti/aligned-model-v2-fin/zephyr_aligned_DPO_v2_2_fin/Zephyr_Manual_DPO_Adapter_v2_2"
battute = generate_for_adapter(BASE_ZEPHYR_ID, tokenizer_zephyr, PATH_DPO_V2_2, PROMPTS, MASTER_SYSTEM_PROMPT)
salva_colonna("DPO_v2.2", battute)


# ==========================================
# LA NUOVA FASE PER IL MODELLO BASE
# ==========================================
print("\n FASE 3: Zephyr-7B BASE (Zero-Shot)...")

# 1. Carichiamo il tokenizer ORIGINALE e puro
tokenizer_base = AutoTokenizer.from_pretrained(BASE_ZEPHYR_ID)
tokenizer_base.pad_token = tokenizer_base.eos_token

# 2. Generiamo usando il tokenizer base e il modello base
battute_base = generate_for_base_model(BASE_ZEPHYR_ID, tokenizer_base, PROMPTS, MASTER_SYSTEM_PROMPT)

# 3. Salviamo nella nuova colonna
salva_colonna("Zephyr_Base", battute_base)

print("\n DATASET GENERATO CON BASE INCLUSA! ")



 FASE 1: Mistral-7B...



---> Caricamento Modello Base: mistralai/Mistral-7B-Instruct-v0.2


config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

...

special_tokens_map.json:   0%|          | 0.00/168 [00:00<?, ?B/s]

  -> Caricamento Modello Base (Senza Adapter): HuggingFaceH4/zephyr-7b-beta


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Generazione Zero-Shot: 100%|██████████| 50/50 [02:24<00:00,  2.90s/it]


 Checkpoint salvato per Zephyr_Base!

 DATASET GENERATO CON BASE INCLUSA! 
